# Train & Fine-Tune Road Following Model -> Export to ONNX

This notebook loads previously saved image datasets (`road_following_A`, `road_following_B`, etc.), automatically checks if an existing trained PyTorch model (`road_following_model.pth`) exists for **Fine-Tuning**, trains/fine-tunes the ResNet-18 model, and exports the final updated weights directly into an **ONNX model file** (`road_following_model.onnx`) for high-performance inference on JetRacer.

| Step | Description |
|------|-------------|
| 1 | Setup Environment & Load Saved Datasets |
| 2 | Initialize Model & Load Existing Weights (Fine-Tuning) |
| 3 | Train / Fine-Tune Model & Export to `.onnx` Format |
| 4 | Verify Exported ONNX Model with ONNX Runtime |

### 1. Setup Environment & Load Saved Datasets

In [1]:
import os
import sys
import glob
import time
import torch
import torchvision
import cv2
import numpy as np
from pathlib import Path

# Add parent directory to sys.path
parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.append(str(parent_dir))

try:
    from jetracer.notebooks.xy_dataset import XYDataset
except ImportError:
    from xy_dataset import XYDataset

TASK = 'road_following'
CATEGORIES = ['apex']

# Find all saved dataset folders matching 'road_following_*'
dataset_dirs = glob.glob(os.path.join(Path.cwd(), f"{TASK}_*"))
if not dataset_dirs:
    dataset_dirs = glob.glob(os.path.join(parent_dir, "notebooks", f"{TASK}_*"))

print(f"[*] Found {len(dataset_dirs)} dataset directories:")
for d in dataset_dirs:
    print(f"  - {os.path.basename(d)}")

# Load and combine all dataset folders
datasets = []
total_samples = 0
for d in dataset_dirs:
    folder_name = os.path.basename(d)
    ds = XYDataset(d, CATEGORIES, random_hflip=True)
    datasets.append(ds)
    print(f"  [+] Loaded '{folder_name}': {len(ds)} samples")
    total_samples += len(ds)

if len(datasets) == 1:
    dataset = datasets[0]
elif len(datasets) > 1:
    dataset = torch.utils.data.ConcatDataset(datasets)
    dataset.categories = CATEGORIES
else:
    print("[!] WARNING: No dataset folders found. Collect data samples using interactive_regression.ipynb first.")
    dataset = None

if dataset:
    print(f"\n[+] Total combined training dataset size: {total_samples} samples!")


[*] Found 7 dataset directories:
  - road_following_A
  - road_following_live.ipynb
  - road_following_model.onnx
  - road_following_model.pth
  - road_following_stanley_onnx.py
  - road_following_stanley_onnx_ros.py
  - road_following_stanley_pth.py
  [+] Loaded 'road_following_A': 104 samples
  [+] Loaded 'road_following_live.ipynb': 0 samples
  [+] Loaded 'road_following_model.onnx': 0 samples
  [+] Loaded 'road_following_model.pth': 0 samples
  [+] Loaded 'road_following_stanley_onnx.py': 0 samples
  [+] Loaded 'road_following_stanley_onnx_ros.py': 0 samples
  [+] Loaded 'road_following_stanley_pth.py': 0 samples

[+] Total combined training dataset size: 104 samples!


### 2. Initialize Model & Check Existing Weights for Fine-Tuning

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dim = 2 * len(CATEGORIES)  # (x, y) coordinates

pth_save_path  = os.path.join(Path.cwd(), "road_following_model.pth")
onnx_save_path = os.path.join(Path.cwd(), "road_following_model.onnx")

# 1. Base ResNet-18 Architecture
model = torchvision.models.resnet18(pretrained=True)
model.fc = torch.nn.Linear(512, output_dim)
model = model.to(device)

# 2. Fine-Tuning Logic: Check if existing model weights exist!
if os.path.exists(pth_save_path):
    try:
        model.load_state_dict(torch.load(pth_save_path, map_location=device))
        print(f"=======================================================")
        print(f"   [+] FINE-TUNING MODE ACTIVATED!                      ")
        print(f"   Successfully loaded existing weights:               ")
        print(f"   {pth_save_path}                                     ")
        print(f"=======================================================")
    except Exception as e:
        print(f"[!] Could not load existing model weights ({e}). Initializing fresh weights.")
else:
    print(f"=======================================================")
    print(f"   [*] NEW MODEL INITIALIZED (ImageNet Pretrained)     ")
    print(f"   No existing weights file found at:                  ")
    print(f"   {pth_save_path}                                     ")
    print(f"=======================================================")

print(f"[*] PyTorch weights save path: {pth_save_path}")
print(f"[*] ONNX export path         : {onnx_save_path}")


d:\Users\nhatt\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Users\nhatt\AppData\Local\Programs\Python\Python314\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


   [+] FINE-TUNING MODE ACTIVATED!                      
   Successfully loaded existing weights:               
   d:\JetRacer_AI\jetracer\notebooks\road_following_model.pth                                     
[*] PyTorch weights save path: d:\JetRacer_AI\jetracer\notebooks\road_following_model.pth
[*] ONNX export path         : d:\JetRacer_AI\jetracer\notebooks\road_following_model.onnx


### 3. Fine-Tune PyTorch Model & Export Directly to `.onnx` Format

In [3]:
def train_and_export(epochs=10, batch_size=8, lr=1e-3):
    if dataset is None or total_samples == 0:
        print("[!] ERROR: Dataset is empty! Save data samples before training.")
        return

    train_loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    mode_name = "FINE-TUNING" if os.path.exists(pth_save_path) else "TRAINING"
    print(f"=======================================================")
    print(f"   STARTING {mode_name} ({epochs} Epochs, Batch Size {batch_size})   ")
    print(f"=======================================================\n")
    
    model.train()
    start_time = time.time()

    for epoch in range(epochs):
        sum_loss = 0.0
        count = 0
        for images, category_idx, xy in train_loader:
            images = images.to(device)
            xy = xy.to(device)

            optimizer.zero_grad()
            outputs = model(images)

            loss = 0.0
            for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                loss += torch.mean((outputs[batch_idx][2 * cat_idx:2 * cat_idx + 2] - xy[batch_idx]) ** 2)
            loss /= len(category_idx)

            loss.backward()
            optimizer.step()

            sum_loss += float(loss) * len(category_idx)
            count += len(category_idx)

        avg_loss = sum_loss / count
        print(f"  Epoch [{epoch+1:02d}/{epochs:02d}] - Loss: {avg_loss:.4f}")

    elapsed = time.time() - start_time
    print(f"\n[+] Fine-Tuning completed in {elapsed:.1f}s!")

    # 1. Save updated PyTorch Model (.pth)
    model.eval()
    torch.save(model.state_dict(), pth_save_path)
    print(f"[+] Saved updated PyTorch model -> {pth_save_path}")

    # 2. Export updated model directly to ONNX format (.onnx)
    print(f"[*] Exporting Fine-Tuned PyTorch model to ONNX format...")
    dummy_input = torch.randn(1, 3, 224, 224, device=device)
    
    try:
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            verbose=False,
            input_names=['input_0'],
            output_names=['output_0'],
            dynamo=False
        )
    except Exception:
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            verbose=False,
            input_names=['input_0'],
            output_names=['output_0']
        )
    
    if os.path.exists(onnx_save_path):
        size_mb = os.path.getsize(onnx_save_path) / (1024 * 1024)
        print(f"=======================================================")
        print(f"   SUCCESSFULLY EXPORTED FINE-TUNED ONNX MODEL!        ")
        print(f"   File Path : {onnx_save_path}                         ")
        print(f"   File Size : {size_mb:.2f} MB                        ")
        print(f"=======================================================")
    else:
        print(f"[!] ERROR: Failed to create ONNX file at '{onnx_save_path}'")

# Run fine-tuning & ONNX export:
train_and_export(epochs=10, batch_size=8, lr=1e-3)


   STARTING FINE-TUNING (10 Epochs, Batch Size 8)   



C:\Users\nhatt\AppData\Local\Temp\ipykernel_15332\1903948734.py:41: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  sum_loss += float(loss) * len(category_idx)


  Epoch [01/10] - Loss: 0.1366
  Epoch [02/10] - Loss: 0.0494
  Epoch [03/10] - Loss: 0.0479
  Epoch [04/10] - Loss: 0.0371
  Epoch [05/10] - Loss: 0.0184
  Epoch [06/10] - Loss: 0.0105
  Epoch [07/10] - Loss: 0.0165
  Epoch [08/10] - Loss: 0.0131
  Epoch [09/10] - Loss: 0.0130
  Epoch [10/10] - Loss: 0.0263

[+] Fine-Tuning completed in 9.9s!
[+] Saved updated PyTorch model -> d:\JetRacer_AI\jetracer\notebooks\road_following_model.pth
[*] Exporting Fine-Tuned PyTorch model to ONNX format...


C:\Users\nhatt\AppData\Local\Temp\ipykernel_15332\1903948734.py:60: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


   SUCCESSFULLY EXPORTED FINE-TUNED ONNX MODEL!        
   File Path : d:\JetRacer_AI\jetracer\notebooks\road_following_model.onnx                         
   File Size : 42.63 MB                        


### 4. Verify Exported ONNX Model with ONNX Runtime

In [ ]:
import onnxruntime as ort

print("[*] Verifying exported ONNX model with ONNX Runtime...")
try:
    session = ort.InferenceSession(onnx_save_path, providers=['CPUExecutionProvider'])
    input_info = session.get_inputs()[0]
    output_info = session.get_outputs()[0]
    
    print(f"  [+] Input Name  : {input_info.name} (Shape: {input_info.shape}, Type: {input_info.type})")
    print(f"  [+] Output Name : {output_info.name} (Shape: {output_info.shape}, Type: {output_info.type})")
    
    # Test dummy inference
    dummy_np = np.random.randn(1, 3, 224, 224).astype(np.float32)
    outs = session.run([output_info.name], {input_info.name: dummy_np})
    print(f"  [+] Test Inference Output: {outs[0].flatten()}")
    print("\n[+] ONNX Model Verification PASSED 100%! Ready for road_following_live.ipynb!")
except Exception as e:
    print(f"[!] ONNX Verification Exception: {e}")
